**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**09 | HARTH Source Quality and Minority-Activity Audit**

![HARTH source quality and minority activity workflow](../../images/09_consolidated_workflow.svg)

**Objective:** Audit original HARTH v2.0 annotation codes, timestamp continuity, eligible windows, participant-level activity support, annotation-bout durations, sampling caps and six-channel acceleration measurements. Explain what evidence exists for minority activities without inferring the cause of model errors.

**Dataset:** Pinned official NTNU AI Lab HARTH v2.0 source recordings. The audit covers **29 Development participants**, excluding S023/S032. Only eight documented task activities are eligible for classification; original other/undocumented codes are preserved without invented meanings. Candidate windows use 100 samples and a 50-sample stride at nominal 50 Hz.

**Analyses:** Original source labels and sensor descriptors; mutually exclusive label/timestamp window rejections; eligible activity-window and unique-participant support; annotation bouts; independently selected label-blind per-participant sampling caps 250/500/1,000; and actual source-window examples. This is a **model-free source audit**.

**Model-error context:** Read the current saved `08_confidence_class_error_summary.csv` from Notebook 08. Full eligible source windows and capped out-of-fold model predictions have different denominators; their descriptive comparison does not establish causality or independent model evaluation.

**Scientific boundary:** No model is trained, no Test subject is loaded, and no fresh performance results are asserted before integrated execution. Stage-specific CSV/PNG files will use the `09_` prefix.


**00. Colab Repository Setup**

Use the same Colab repository setup as the previous integrated notebooks.


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Source-Audit Environment**

Load the current HARTH v2.0 source adapter, source-quality and temporal sampling helpers, and configure the shared output directory. No model is fitted.


In [ ]:
# ================= 01. Imports and Source-Audit Environment ==================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4),
)

# -------------------- 4. HARTH Source and Audit Utilities --------------------
from core.output_download import download_outputs_from_colab
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, HARTH_REVISION
from harth_quality import audit_harth_development_sources
from core.sampling_audit import audit_temporal_sampling

# ----------------------- 5. Source and Output Settings -----------------------
RESERVED_TEST = {"S023", "S032"}
SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
CLASSES = tuple(range(1, 9))
DATA_DIR = root / "data/raw/harth_v2"
output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

print("Development participants:", len(SUBJECTS))
print("Task classes:", CLASSES)
print("Pinned revision:", HARTH_REVISION)
print("Source audit output directory:", output_dir)


**02. Audit Each Development Recording**

Count raw labels and timestamp intervals independently for all 29 Development participants, construct label-pure continuous 100-sample windows with stride 50 and document eligible/out-of-task windows.


In [ ]:
# =================== 02. Audit Each Development Recording ====================

# ---------------- 1. Audit Individual Development Recordings -----------------
(overview, raw_codes, window_codes, metadata,
 bouts, channels, examples) = audit_harth_development_sources(
    subjects=SUBJECTS, data_dir=DATA_DIR, task_classes=CLASSES,
    window_size=100, step_size=50, sampling_rate=50,
    reserved_subjects=RESERVED_TEST,
)

# ------------------- 2. Reconcile Source and Window Counts -------------------
display(overview.sort_values("subject_id"))
print("\n" + "=" * 65)
print("DEVELOPMENT SOURCE ACCOUNTING")
print("=" * 65)
print("Source participants:", overview["subject_id"].nunique())
print("Original CSV rows:", f'{overview["raw_samples"].sum():,}')
print("All valid pure-label windows:", f'{overview["retained_pure_time_safe"].sum():,}')
print("Eight-class eligible windows:", f'{len(metadata):,}')
print("Windows excluded after segmentation:", f'{overview["excluded_after_windowing"].sum():,}')

if set(metadata["subject_id"]) != set(SUBJECTS):
    raise ValueError("Development participant coverage is incomplete.")
if len(metadata) != int(overview["eligible_eight_class_windows"].sum()):
    raise ValueError("Source eligible window counts are inconsistent.")


**03. Distinguish Documented and Unmapped Source Codes**

Preserve all original annotations. Distinguish documented eight-class task activities from other documented codes and unmapped codes, without assigning meaning to codes 10, 11 or 17.


In [ ]:
# =========== 03. Distinguish Documented and Unmapped Source Codes ============

# --------------------- 1. Inspect Original Source Labels ---------------------
source_labels = raw_codes.groupby(
    ["source_code", "source_activity", "in_eight_class_task"],
    as_index=False
)["source_samples"].sum().sort_values("source_code")

display(source_labels)
excluded_codes = raw_codes.loc[
    ~raw_codes["in_eight_class_task"]
].sort_values(["source_code", "subject_id"])

display(excluded_codes[[
    "subject_id", "source_code", "source_activity",
    "source_samples", "documented_source_code"
]])

# ----------------- 2. Report Excluded and Undocumented Codes -----------------
post_window = window_codes.groupby(
    ["source_code", "source_activity", "in_eight_class_task"],
    as_index=False
)["accepted_pure_windows"].sum().sort_values("source_code")

display(post_window)
unknown_codes = sorted(
    raw_codes.loc[~raw_codes["documented_source_code"], "source_code"].unique().tolist()
)

print("Undocumented source codes observed:", unknown_codes)
print("Unknown-code semantic interpretation performed: NO")


**04. Compare Windows and Independent Participant Support**

Measure full eligible windows and distinct people per activity. Participant support is not the same as the number of overlapping source windows.


In [ ]:
# ========== 04. Compare Windows and Independent Participant Support ==========

# ------------------- 1. Compute Participant-Class Coverage -------------------
coverage = pd.crosstab(metadata["subject_id"], metadata["label"]).reindex(
    index=SUBJECTS, columns=CLASSES, fill_value=0
).astype(int)
coverage.columns = [ACTIVITY_NAMES[c] for c in CLASSES]

display(coverage)

# ----------------------- 2. Summarise Activity Support -----------------------
class_summary = pd.DataFrame({
    "activity": coverage.columns,
    "eligible_windows": coverage.sum(axis=0).to_numpy(),
    "supporting_subjects": (coverage > 0).sum(axis=0).to_numpy(),
})
class_summary["window_share_pct"] = (
    class_summary["eligible_windows"] / class_summary["eligible_windows"].sum() * 100
)

class_summary["subject_coverage_pct"] = (
    class_summary["supporting_subjects"] / len(SUBJECTS) * 100
)

display(class_summary.round(2))

# ------------------ 3. Visualise Source Window Distribution ------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(class_summary["activity"], class_summary["eligible_windows"])
axes[0].set(title="Eligible windows per activity", xlabel="Overlapping windows")
axes[1].barh(class_summary["activity"], class_summary["supporting_subjects"])
axes[1].set(title="Independent participant support", xlabel="Participants", xlim=(0,29))
for ax in axes:
    ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "09_activity_window_and_subject_support.png")
plt.show()


**05. Visualise Activity Coverage Across Participants**

Compare raw class window counts (log colour scale) with percentages within each participant. These are observed recording frequencies, not population prevalence.


In [ ]:
# ============ 05. Visualise Activity Coverage Across Participants ============

# ----------- 1. Calculate Within-Participant Activity Proportions ------------
fractions = coverage.div(coverage.sum(axis=1), axis=0).mul(100)

# ---------------- 2. Visualise Absolute and Relative Coverage ----------------
fig, axes = plt.subplots(1, 2, figsize=(16, 10), sharey=True)

maps = [
    (np.log1p(coverage.to_numpy()), "log(1 + windows)", "Window counts (log scale)"),
    (fractions.to_numpy(), "Percent of participant windows", "Within-person activity (%)"),
]

for ax, (matrix, unit, title) in zip(axes, maps):
    handle = ax.imshow(matrix, aspect="auto", interpolation="nearest")
    ax.set_xticks(range(len(CLASSES)), coverage.columns, rotation=70, ha="right")
    ax.set_yticks(range(len(SUBJECTS)), SUBJECTS)
    ax.set(title=title, xlabel="Documented activity")
    fig.colorbar(handle, ax=ax, label=unit, fraction=0.046, pad=0.02)

axes[0].set_ylabel("Development participant")
fig.tight_layout()
save_figure(fig, output_dir / "09_participant_activity_coverage.png")
plt.show()


**06. Inspect Annotation-Bout Durations**

Inspect nominal annotation-bout durations and the proportion shorter than a two-second window, focusing on Shuffling and the two stair activities.


In [ ]:
# =================== 06. Inspect Annotation-Bout Durations ===================

# --------------------- 1. Select Minority-Activity Bouts ---------------------
task_bouts = bouts.loc[bouts["source_code"].isin(CLASSES)].copy()
focus = task_bouts.loc[task_bouts["source_code"].isin([3, 4, 5])].copy()

# ------------------------ 2. Summarise Bout Durations ------------------------
bout_summary = task_bouts.groupby("source_code").agg(
    bouts=("samples", "size"),
    median_duration_s=("nominal_duration_seconds", "median"),
    mean_duration_s=("nominal_duration_seconds", "mean"),
    short_bouts=("shorter_than_window", "sum"),
    contributors=("subject_id", "nunique"),
).reindex(CLASSES)
bout_summary["activity"] = [ACTIVITY_NAMES[c] for c in CLASSES]
bout_summary["short_bout_pct"] = bout_summary["short_bouts"] / bout_summary["bouts"] * 100

display(bout_summary.reset_index()[[
    "source_code", "activity", "bouts", "contributors",
    "median_duration_s", "mean_duration_s", "short_bouts", "short_bout_pct"
]].round(3))

# -------------------- 3. Visualise Minority Bout Lengths ---------------------
fig, ax = plt.subplots(figsize=(11, 5))
values = [
    focus.loc[focus["source_code"] == c, "nominal_duration_seconds"]
    for c in [3, 4, 5]
]
ax.boxplot(values, tick_labels=[ACTIVITY_NAMES[c] for c in [3, 4, 5]],
           showfliers=False)

ax.axhline(2, linestyle="--", color="gray", label="100 samples at 50 Hz")
ax.set(title="Minority activity annotation-bout durations",
       ylabel="Nominal duration (seconds)")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "09_minority_annotation_bout_durations.png")
plt.show()


**07. Reconcile Candidate-Window Rejections**

Separate label-only, timestamp-only and joint window rejections to avoid double counting. Timestamp anomalies do not independently prove a sensor defect.


In [ ]:
# ================= 07. Reconcile Candidate-Window Rejections =================

# ----------------- 1. Audit Window Acceptance and Rejection ------------------
reasons = [
    "retained_pure_time_safe", "rejected_label_only",
    "rejected_time_only", "rejected_both"
]
reason_summary = pd.DataFrame({
    "decision": reasons,
    "windows": [int(overview[name].sum()) for name in reasons]
})

# -------------- 2. Examine Participant-Level Rejection Patterns --------------
reason_summary["share_pct"] = (
    reason_summary["windows"] / overview["candidate_windows"].sum() * 100
)

display(reason_summary.round(2))
per_subject_rejection = overview[[
    "subject_id", "candidate_windows",
    "rejected_label_only", "rejected_time_only", "rejected_both",
    "nonpositive_intervals", "long_intervals",
    "median_positive_interval_seconds"
]].copy()
per_subject_rejection["rejection_pct"] = (
    (per_subject_rejection[["rejected_label_only",
      "rejected_time_only", "rejected_both"]].sum(axis=1))
    / per_subject_rejection["candidate_windows"] * 100
)

display(per_subject_rejection.sort_values("rejection_pct", ascending=False).round(3))

# ---------------------- 3. Visualise Rejection Reasons -----------------------
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(reason_summary["decision"], reason_summary["windows"])
ax.set(title="Mutually exclusive source-window decisions", ylabel="Candidate windows")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "09_source_window_rejection_reasons.png")
plt.show()


**08. Audit Label-Independent Sampling Caps**

Compare independently sampled per-person caps of 250, 500 and 1,000 with full eligible windows. Unlike nested training cohorts, these cap selections are not required to include smaller sets.


In [ ]:
# ================= 08. Audit Label-Independent Sampling Caps =================

# ----------------- 1. Compare Label-Blind Sampling Policies ------------------
selected, per_subject, cap_counts = audit_temporal_sampling(
    metadata=metadata, caps=(250, 500, 1000),
    classes=CLASSES, forbidden_subjects=RESERVED_TEST,
)
cap_order = ["full", "cap_250", "cap_500", "cap_1000"]
count_table = cap_counts.pivot(
    index="label", columns="policy", values="windows"
).reindex(index=CLASSES, columns=cap_order).astype(int)

# ---------------------- 2. Calculate Activity Retention ----------------------
retention = count_table.div(count_table["full"], axis=0) * 100
retention.index = [ACTIVITY_NAMES[c] for c in CLASSES]

display(count_table)
display(retention.round(2))
support_table = selected.groupby(["label", "policy"])["subject_id"].nunique().unstack(
    fill_value=0
).reindex(index=CLASSES, columns=cap_order, fill_value=0).astype(int)
support_table.index = [ACTIVITY_NAMES[c] for c in CLASSES]

display(support_table)

# ------------------- 3. Visualise Minority-Class Retention -------------------
minority_names = [ACTIVITY_NAMES[c] for c in (3, 4, 5)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

retention.loc[minority_names, ["cap_250","cap_500","cap_1000"]].plot.bar(
    ax=axes[0], rot=15, title="Minority-class window retention (%)"
)

support_table.loc[minority_names, ["cap_250","cap_500","cap_1000"]].plot.bar(
    ax=axes[1], rot=15, title="Independent participants retained"
)
axes[0].set(ylabel="Percent of full eligible windows", ylim=(0, 105))
axes[1].set(ylabel="Participants", ylim=(0, 29))
for ax in axes:
    ax.grid(axis="y", alpha=0.2)

fig.tight_layout()
save_figure(fig, output_dir / "09_sampling_minority_coverage.png")
plt.show()


**09. Inspect Source Sensor Channels and Examples**

Characterise six-axis extrema, dispersion, repeated neighbouring samples and non-finite counts; show original source windows with participant and source-index provenance.


In [ ]:
# ============== 09. Inspect Source Sensor Channels and Examples ==============

# ------------------ 1. Inspect Sensor Ranges and Continuity ------------------
channel_summary = channels.groupby("channel").agg(
    recordings=("subject_id", "nunique"),
    median_std=("std", "median"),
    median_absolute_p99=("absolute_p99", "median"),
    largest_absolute_max=("maximum", "max"),
    smallest_minimum=("minimum", "min"),
    mean_repeated_adjacent_pct=("repeated_adjacent_pct", "mean"),
    nonfinite_samples=("nonfinite_samples", "sum"),
)

display(channel_summary.round(4))
display(channels.sort_values("repeated_adjacent_pct", ascending=False).head(12).round(4))

# ---------------- 2. Visualise Original Accelerometer Signals ----------------
chosen_codes = [1, 3, 4, 5]

fig, axes = plt.subplots(len(chosen_codes), 1, figsize=(13, 12), sharex=True)
for ax, code in zip(axes, chosen_codes):
    item = examples.get(code)
    if item is None:
        ax.text(.5, .5, "No eligible source window", ha="center", va="center")
        continue
    for index, name in enumerate(item["channel_names"]):
        ax.plot(np.arange(item["signals"].shape[1]) / 50,
                item["signals"][index], label=name, linewidth=1)
    ax.set_title(f'{ACTIVITY_NAMES[code]} | {item["subject_id"]} | '
                 f'source index {item["start_index"]}')
    ax.set_ylabel("Raw source value")
    ax.grid(alpha=0.15)

axes[0].legend(ncol=3, fontsize=8)
axes[-1].set_xlabel("Time within source window (seconds)")
fig.tight_layout()
save_figure(fig, output_dir / "09_acceleration_source_examples.png")
plt.show()


**10. Relate Source Support to Current OOF Errors**

Compare full source support with read-only current Notebook 08 class-error evidence from base-cap held-out predictions. They have different denominators; association is not causation.


In [ ]:
# ============== 10. Relate Source Support to Current OOF Errors ==============

# ----------------- 1. Read Current Held-Out Class Error Data -----------------
evidence_path = output_dir / "08_confidence_class_error_summary.csv"
current_errors = pd.read_csv(evidence_path)
required_cols = {
    "class", "support", "recall", "errors",
    "high_confidence_errors_0_90",
}
if not required_cols.issubset(current_errors.columns):
    raise ValueError("Notebook 08 source class-error columns are missing.")
if set(current_errors["class"]) != set(class_summary["activity"]):
    raise ValueError("Task activity labels differ from saved OOF evidence.")

# -------------- 2. Join Distinct Source/Prediction Denominators --------------
source_model_context = class_summary.merge(
    current_errors[[
        "class", "support", "recall", "errors",
        "high_confidence_errors_0_90",
    ]],
    left_on="activity", right_on="class",
    how="left", validate="one_to_one",
).drop(columns="class").rename(columns={
    "eligible_windows": "full_eligible_windows",
    "supporting_subjects": "full_source_contributors",
    "support": "oof_base_windows",
    "recall": "oof_recall",
    "errors": "oof_errors",
})
display(source_model_context.round(4))
minority = [ACTIVITY_NAMES[c] for c in (3, 4, 5)]
display(source_model_context.loc[
    source_model_context["activity"].isin(minority)
].round(4))
print("Source coverage does not establish the cause of model errors.")


**11. Export Source-Quality Research Evidence**

Export and package source labels, window acceptance, participant coverage, annotation bouts, independent cap effects, sensor statistics and source-to-model error context.


In [ ]:
# ================ 11. Export Source-Quality Research Evidence ================

# ----------------- 1. Export Source-Quality Research Outputs -----------------
overview.to_csv(output_dir / "09_development_source_accounting.csv", index=False)
raw_codes.to_csv(output_dir / "09_raw_source_label_counts.csv", index=False)
window_codes.to_csv(output_dir / "09_accepted_window_source_codes.csv", index=False)
source_labels.to_csv(output_dir / "09_source_label_summary.csv", index=False)
excluded_codes.to_csv(output_dir / "09_excluded_source_codes.csv", index=False)
post_window.to_csv(output_dir / "09_post_window_source_labels.csv", index=False)
# -------------- 2. Save Participant, Window and Sensor Evidence --------------
coverage.rename_axis("subject_id").reset_index().to_csv(
    output_dir / "09_participant_activity_counts.csv", index=False
)
fractions.rename_axis("subject_id").reset_index().to_csv(
    output_dir / "09_participant_activity_percentages.csv", index=False
)
class_summary.to_csv(output_dir / "09_activity_support_summary.csv", index=False)
bouts.to_csv(output_dir / "09_source_annotation_bouts.csv", index=False)
bout_summary.reset_index().to_csv(
    output_dir / "09_annotation_bout_summary.csv", index=False
)
reason_summary.to_csv(output_dir / "09_window_rejection_summary.csv", index=False)
per_subject_rejection.to_csv(
    output_dir / "09_participant_window_rejections.csv", index=False
)
per_subject.to_csv(output_dir / "09_sampling_subject_totals.csv", index=False)
cap_counts.to_csv(output_dir / "09_sampling_class_counts.csv", index=False)
retention.rename_axis("activity").reset_index().to_csv(
    output_dir / "09_sampling_activity_retention_pct.csv", index=False
)
support_table.rename_axis("activity").reset_index().to_csv(
    output_dir / "09_sampling_participant_support.csv", index=False
)
channels.to_csv(output_dir / "09_sensor_channel_descriptives.csv", index=False)
channel_summary.rename_axis("channel").reset_index().to_csv(
    output_dir / "09_sensor_channel_summary.csv", index=False
)
# Preserve original source-window indices for independent inspection.
metadata.to_csv(output_dir / "09_eligible_window_metadata.csv", index=False)
source_model_context.to_csv(
    output_dir / "09_source_support_and_oof_error_context.csv", index=False
)
print("Development participants:", len(SUBJECTS))
print("Eligible task windows:", f"{len(metadata):,}")
print("Task activities:", len(CLASSES))
print("Models fitted in this notebook: none")
print("Source-quality research tables saved in:", output_dir)

# --------------------- 3. Package Current Stage Outputs ----------------------
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="09"
)


**12. Scientific Interpretation and Research Boundaries**

**Source integrity:** Preserve the original HARTH annotations and time order. Only label-pure, time-continuous source windows enter the documented eight-class task. Label-only, time-only and joint rejection counts are mutually exclusive; unknown source labels retain their original codes.

**Minority-class support:** The number of overlapping windows and the number of independent participants are different units. Variation in Shuffling and stair-class availability, annotation-bout duration and participant representation constrains what model comparisons can establish. A short annotation bout or unusual sensor range is not proof of inaccurate ground truth or device failure.

**Sampling:** Independent per-participant caps of 250, 500 and 1,000 windows may remove activity evidence or contributing participants. Window retention percentages are descriptive source quantities, not classification sensitivity, and they are not the strictly nested training-set expansion used in other notebooks.

**Contextual model errors:** Current Notebook 08 out-of-fold error metrics concern base-capped Validation windows, whereas this audit measures full eligible source windows. Their descriptive association cannot prove a causal mechanism. No classifier or scaler is fitted here, and no independent Test observations are accessed.

**Results:** Add numerical interpretation after Notebook 09 has executed and its current `09_` outputs have been inspected.
